# BERNN minimal tutorials (4 examples)

This notebook provides the 4 shortest runnable examples:

1. `TrainAEClassifierHoldout` + `pools=False`
2. `TrainAEClassifierHoldout` + `pools=True`
3. `TrainAEThenClassifierHoldout` + `pools=False`
4. `TrainAEThenClassifierHoldout` + `pools=True`

Dataset used: `../data/benchmark/intensities.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from bernn import TrainAEClassifierHoldout, TrainAEThenClassifierHoldout
from bernn.config.training_config import TrainingConfig

csv_path = Path('../data/benchmark/intensities.csv')
if not csv_path.exists():
    raise FileNotFoundError(f'Missing dataset: {csv_path.resolve()}')

df = pd.read_csv(csv_path)
X = df.iloc[:, 3:]
y = df.iloc[:, 1].to_numpy()
batches = df.iloc[:, 2].to_numpy()

split = max(8, int(0.8 * len(df)))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y[:split], y[split:]
batches_train, batches_test = batches[:split], batches[split:]

print('train:', X_train.shape, 'test:', X_test.shape)

In [ ]:
# Minimal fast config for smoke-level runs
bernn_config = TrainingConfig(
    optimize_hyperparams=False,
    n_trials=1,
    n_repeats=1,
    n_layers=1,
    layer1=128,
    warmup=1,
    n_epochs=2,
    dloss='inverseTriplet',
    device='cpu',
    scaler='standard',
    bs=16,
)

In [ ]:
examples = [
    ('classifier_no_pool', TrainAEClassifierHoldout, False),
    ('classifier_with_pool', TrainAEClassifierHoldout, True),
    ('ae_then_no_pool', TrainAEThenClassifierHoldout, False),
    ('ae_then_with_pool', TrainAEThenClassifierHoldout, True),
]

predictions = {}
for name, trainer_cls, pools in examples:
    trainer = trainer_cls(config=bernn_config, pools=pools, log_metrics=True, keep_models=False)

    _ = trainer.fit_predict(
        X_train,
        y_train,
        X_test=X_test,
        y_test=y_test,
        groups_train=batches_train,
        groups_test=batches_test,
        cross_validation=False,
        cross_test=False,
    )

    predictions[name] = trainer.predict(X_test)
    print(name, 'ok', 'pred_len=', len(predictions[name]))